# Bài 4: K láng giềng gần nhất - sáu bài tập

- **Môn học:** Học máy và ứng dụng
- **Sinh viên:** Huỳnh Hữu Phúc
- **MSSV:** 2474802010314

Mỗi bài sử dụng dữ liệu và cách chia tập của tài liệu Lab 4, in kết quả cần đối chiếu và có nhận xét ngay bên dưới.

## Bài tập 1. Tự viết KNN không dùng thư viện

Viết hàm `knn_du_doan`, dự đoán toàn bộ tập kiểm tra hai vầng trăng với `k=13` và so với scikit-learn.

In [1]:
# -*- coding: utf-8 -*-
"""Bài 1: tự viết KNN bằng NumPy và đối chiếu với scikit-learn."""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Trả về nhãn được đa số trong k láng giềng gần nhất bỏ phiếu."""
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    chi_so_gan = np.argsort(khoang_cach)[:k]
    so_phieu = np.bincount(y_hoc[chi_so_gan].astype(int))
    return int(np.argmax(so_phieu))


df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

du_doan_tu_viet = np.array([
    knn_du_doan(X_tr.to_numpy(), y_tr.to_numpy(), x_moi, 13)
    for x_moi in X_te.to_numpy()
])
mo_hinh = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
du_doan_thu_vien = mo_hinh.predict(X_te)

acc_tu_viet = accuracy_score(y_te, du_doan_tu_viet)
acc_thu_vien = accuracy_score(y_te, du_doan_thu_vien)
so_khac = int((du_doan_tu_viet != du_doan_thu_vien).sum())

print(f"Accuracy KNN tu viet       : {acc_tu_viet:.4f}")
print(f"Accuracy scikit-learn      : {acc_thu_vien:.4f}")
print(f"So du doan khac nhau       : {so_khac}")
assert so_khac == 0

Accuracy KNN tu viet       : 0.9200
Accuracy scikit-learn      : 0.9200
So du doan khac nhau       : 0


**Nhận xét:** KNN tự viết và `KNeighborsClassifier` cùng đạt accuracy `0,9200`; cả 75 dự đoán đều trùng nhau. Hàm tự viết thực hiện đúng ba bước: tính khoảng cách, lấy `k` chỉ số nhỏ nhất và bỏ phiếu đa số.

## Bài tập 2. Khoảng cách Manhattan

Chọn `k` bằng kiểm định chéo năm phần với khoảng cách Manhattan, rồi so với Euclid trên cùng cách chia dữ liệu.

In [2]:
# -*- coding: utf-8 -*-
"""Bài 2: chọn k với Manhattan và so sánh với Euclid."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def chon_k(metric):
    diem = {}
    for k in range(1, 52, 2):
        mo_hinh = KNeighborsClassifier(n_neighbors=k, metric=metric)
        diem[k] = cross_val_score(mo_hinh, X_tr, y_tr, cv=5).mean()
    k_tot = max(diem, key=diem.get)
    mo_hinh = KNeighborsClassifier(n_neighbors=k_tot, metric=metric).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mo_hinh.predict(X_te))
    return k_tot, diem[k_tot], acc


print("Khoang cach      k tot   CV trung binh   test accuracy")
for ten, metric in [("Euclid", "euclidean"), ("Manhattan", "manhattan")]:
    k_tot, diem_cv, acc = chon_k(metric)
    print(f"{ten:12s} {k_tot:5d}        {diem_cv:.4f}          {acc:.4f}")

Khoang cach      k tot   CV trung binh   test accuracy


Euclid          13        0.8667          0.9200


Manhattan       27        0.8667          0.9067


**Nhận xét:** Manhattan chọn `k=27` và đạt test accuracy `0,9067`; Euclid chọn `k=13` và đạt `0,9200`. Chênh lệch chỉ `0,0133`, nên đổi cách đo khoảng cách không làm kết quả thay đổi nhiều trên dữ liệu này.

## Bài tập 3. Đổi đơn vị đo

Đổi chi tiêu từ nghìn đồng sang triệu đồng, so KNN không chuẩn hóa và kiểm tra tính bất biến khi có chuẩn hóa.

In [3]:
# -*- coding: utf-8 -*-
"""Bài 3: đổi đơn vị chi tiêu và đo ảnh hưởng trước/sau chuẩn hóa."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X_nghin = df[["gio_on", "diem_giua_ky", "chi_tieu"]].copy()
y = df["qua_mon"]
X_tr_nghin, X_te_nghin, y_tr, y_te = train_test_split(
    X_nghin, y, test_size=0.25, random_state=17, stratify=y
)

X_tr_trieu = X_tr_nghin.copy()
X_te_trieu = X_te_nghin.copy()
X_tr_trieu["chi_tieu"] = X_tr_trieu["chi_tieu"] / 1000
X_te_trieu["chi_tieu"] = X_te_trieu["chi_tieu"] / 1000

knn_nghin = KNeighborsClassifier(n_neighbors=7).fit(X_tr_nghin, y_tr)
knn_trieu = KNeighborsClassifier(n_neighbors=7).fit(X_tr_trieu, y_tr)
acc_nghin = accuracy_score(y_te, knn_nghin.predict(X_te_nghin))
acc_trieu = accuracy_score(y_te, knn_trieu.predict(X_te_trieu))

def knn_chuan():
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))

mh_chuan_nghin = knn_chuan().fit(X_tr_nghin, y_tr)
mh_chuan_trieu = knn_chuan().fit(X_tr_trieu, y_tr)
acc_chuan_nghin = accuracy_score(y_te, mh_chuan_nghin.predict(X_te_nghin))
acc_chuan_trieu = accuracy_score(y_te, mh_chuan_trieu.predict(X_te_trieu))

print(f"Khong chuan hoa, don vi nghin dong : {acc_nghin:.4f}")
print(f"Khong chuan hoa, don vi trieu dong : {acc_trieu:.4f}")
print(f"Co chuan hoa, don vi nghin dong    : {acc_chuan_nghin:.4f}")
print(f"Co chuan hoa, don vi trieu dong    : {acc_chuan_trieu:.4f}")

Khong chuan hoa, don vi nghin dong : 0.5333
Khong chuan hoa, don vi trieu dong : 0.8667
Co chuan hoa, don vi nghin dong    : 0.8667
Co chuan hoa, don vi trieu dong    : 0.8667


**Nhận xét:** Chỉ đổi đơn vị đã làm accuracy không chuẩn hóa tăng từ `0,5333` lên `0,8667`, vì phép đổi làm cột chi tiêu nhỏ đi 1.000 lần và thay đổi phần đóng góp của nó vào khoảng cách. Khi có `StandardScaler`, hai đơn vị đều đạt `0,8667` vì chuẩn hóa loại bỏ ảnh hưởng của hệ số đổi đơn vị.

## Bài tập 4. Láng giềng có trọng số

So sánh bỏ phiếu đều với bỏ phiếu theo nghịch đảo khoảng cách tại `k=1`, `13` và `101`.

In [4]:
# -*- coding: utf-8 -*-
"""Bài 4: so sánh trọng số đều và trọng số theo khoảng cách."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("  k   weights     tap hoc   tap kiem tra")
for k in [1, 13, 101]:
    for trong_so in ["uniform", "distance"]:
        mo_hinh = KNeighborsClassifier(
            n_neighbors=k, weights=trong_so
        ).fit(X_tr, y_tr)
        acc_hoc = accuracy_score(y_tr, mo_hinh.predict(X_tr))
        acc_kiem_tra = accuracy_score(y_te, mo_hinh.predict(X_te))
        print(f"{k:3d}   {trong_so:8s}      {acc_hoc:.4f}        {acc_kiem_tra:.4f}")

  k   weights     tap hoc   tap kiem tra
  1   uniform       1.0000        0.8000
  1   distance      1.0000        0.8000
 13   uniform       0.8844        0.9200
 13   distance      1.0000        0.9333
101   uniform       0.8311        0.8533
101   distance      1.0000        0.9067


**Nhận xét:** Với `k=1`, hai cách giống nhau. Với `k=13`, trọng số khoảng cách tăng test accuracy từ `0,9200` lên `0,9333`. Với `k=101`, accuracy tăng từ `0,8533` lên `0,9067`: các điểm xa vẫn được xét nhưng có phiếu nhẹ hơn, nên mô hình bớt cẩu thả.

## Bài tập 5. Chọn k cho bài toán hồi quy

Dùng kiểm định chéo năm phần trên tập học để chọn `k=1..20`, sau đó so MAE kiểm tra với hồi quy tuyến tính.

In [5]:
# -*- coding: utf-8 -*-
"""Bài 5: chọn k cho KNN hồi quy bằng MAE kiểm định chéo."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

diem = {}
for k in range(1, 21):
    diem[k] = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_tr,
        y_tr,
        cv=5,
        scoring="neg_mean_absolute_error",
    ).mean()

# Điểm âm càng lớn (càng gần 0) thì MAE càng nhỏ.
k_tot = max(diem, key=diem.get)
print("MAE kiem dinh cheo theo k")
for k in range(1, 21):
    dau = "  <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}  MAE = {-diem[k]:.4f}{dau}")

knn = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print()
print(f"k tot nhat                     : {k_tot}")
print(f"MAE KNN tren tap kiem tra      : {mae_knn:.4f}")
print(f"MAE duong thang tren kiem tra  : {mae_duong:.4f}")

MAE kiem dinh cheo theo k
  k =  1  MAE = 0.4164
  k =  2  MAE = 0.3498  <- tot nhat
  k =  3  MAE = 0.3714
  k =  4  MAE = 0.3618
  k =  5  MAE = 0.3816
  k =  6  MAE = 0.4002
  k =  7  MAE = 0.3990
  k =  8  MAE = 0.4098
  k =  9  MAE = 0.4171
  k = 10  MAE = 0.4365
  k = 11  MAE = 0.4500
  k = 12  MAE = 0.4695
  k = 13  MAE = 0.4992
  k = 14  MAE = 0.5153
  k = 15  MAE = 0.5290
  k = 16  MAE = 0.5595
  k = 17  MAE = 0.5748
  k = 18  MAE = 0.5891
  k = 19  MAE = 0.6130
  k = 20  MAE = 0.6254

k tot nhat                     : 2
MAE KNN tren tap kiem tra      : 0.4867
MAE duong thang tren kiem tra  : 0.2578


**Nhận xét:** Kiểm định chéo chọn `k=2` với CV MAE `0,3498`. Trên tập kiểm tra, KNN đạt MAE `0,4867`, vẫn cao hơn đường thẳng `0,2578`. Quan hệ giá và diện tích gần tuyến tính nên hồi quy tuyến tính phù hợp hơn.

## Bài tập 6. Đo thời gian dự đoán

Đo thời gian `.fit` và `.predict` cho 1.000 mẫu mới khi số mẫu học tăng từ 1.000 đến 100.000.

In [6]:
# -*- coding: utf-8 -*-
"""Bài 6: đo thời gian fit và predict của KNN với ba cỡ dữ liệu."""

import time

import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X_moi = rng.random((1000, 2))
ket_qua = []
so_lan_do = 5

for so_mau in [1000, 10000, 100000]:
    X_hoc = rng.random((so_mau, 2))
    y_hoc = rng.integers(0, 2, size=so_mau)
    cac_lan_fit = []
    for _ in range(so_lan_do):
        mo_hinh = KNeighborsClassifier(n_neighbors=5, algorithm="brute")
        bat_dau = time.perf_counter()
        mo_hinh.fit(X_hoc, y_hoc)
        cac_lan_fit.append(time.perf_counter() - bat_dau)

    # Làm nóng trước khi đo để thời gian khởi tạo thư viện không làm lệch cỡ nhỏ.
    mo_hinh.predict(X_moi[:20])
    cac_lan_predict = []
    for _ in range(so_lan_do):
        bat_dau = time.perf_counter()
        du_doan = mo_hinh.predict(X_moi)
        cac_lan_predict.append(time.perf_counter() - bat_dau)

    thoi_gian_fit = float(np.median(cac_lan_fit))
    thoi_gian_predict = float(np.median(cac_lan_predict))

    ket_qua.append({
        "so_mau_hoc": so_mau,
        "so_mau_du_doan": len(X_moi),
        "fit_giay": thoi_gian_fit,
        "predict_giay": thoi_gian_predict,
        "tong_nhan_1": int(du_doan.sum()),
    })

bang_thoi_gian = pd.DataFrame(ket_qua)
print(bang_thoi_gian.to_string(
    index=False,
    formatters={
        "fit_giay": lambda v: f"{v:.6f}",
        "predict_giay": lambda v: f"{v:.6f}",
    },
))

 so_mau_hoc  so_mau_du_doan fit_giay predict_giay  tong_nhan_1
       1000            1000 0.000458     0.048403          553
      10000            1000 0.002547     0.084627          501
     100000            1000 0.014174     0.478887          496


**Nhận xét:** `.fit` chủ yếu lưu dữ liệu nên rất nhanh. Với `algorithm="brute"`, `.predict` phải tính khoảng cách từ 1.000 mẫu mới tới toàn bộ mẫu học; mốc 100.000 mẫu vì vậy tốn nhiều thời gian hơn rõ rệt. Các mốc nhỏ có thể dao động do chi phí khởi tạo và tối ưu của thư viện. KNN được gọi là mô hình học lười vì phần lớn công việc dồn sang lúc dự đoán.